# 01. EDA
Цель: до любых моделей понять структуру данных, от которой зависят схема разбиения P1/P2/P3 и локальный корпус.
Разделы идут по вопросам, от которых зависит схема валидации. Все функции загрузки и нормализации берутся из `src/data.py`, чтобы предобработка совпадала во всём проекте.

In [1]:
import sys, re
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.compute as pc
from src.data import DATA_DIR, QUERY_KEY, load, norm_text, query_key, content_hash

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
pd.set_option("display.max_colwidth", 80)

## 1. Колонки, типы, пропуски

In [2]:
# Описание читаем отдельно и только ради статистики пропусков:
# это самая тяжёлая колонка, держать её в памяти вместе со всем остальным незачем.
def missing_table(name):
    pf = pq.ParquetFile(DATA_DIR / f"{name}.parquet")
    rows = []
    for field in pf.schema_arrow:
        col = pq.read_table(DATA_DIR / f"{name}.parquet", columns=[field.name]).column(0)
        n_null = col.null_count
        n_empty = 0
        if str(field.type) in ("string", "large_string"):
            # пустая строка или одни пробелы: для текста это тот же пропуск
            n_empty = pc.sum(pc.equal(pc.utf8_trim_whitespace(col), "")).as_py() or 0
        rows.append((field.name, str(field.type), n_null / len(col), n_empty / len(col)))
    out = pd.DataFrame(rows, columns=["column", "type", "nan_share", "empty_str_share"])
    print(f"{name}: {pf.metadata.num_rows} строк")
    return out.round(4)

for name in ["train", "benchmark_queries", "benchmark_items"]:
    display(missing_table(name))

train: 497673 строк


,column,type,nan_share,empty_str_share
0,search_query,string,0.0000,0.0000
1,search_location_id,int64,0.0000,0.0000
2,search_is_delivery_search,int32,0.0000,0.0000
3,search_infm_params_text,string,0.0000,0.3295
4,search_category,int64,0.0000,0.0000
5,item_title_raw,string,0.0000,0.0000
6,item_rating_reviews_count,double,0.0386,0.0000
7,item_rating,double,0.0567,0.0000
8,item_price,"decimal128(27, 15)",0.0000,0.0000
9,item_microcat_id,int64,0.0000,0.0000


benchmark_queries: 2452 строк


,column,type,nan_share,empty_str_share
0,query_id,large_string,0.0,0.0000
1,search_query,large_string,0.0,0.0000
2,search_location_id,int64,0.0,0.0000
3,search_is_delivery_search,int32,0.0,0.0000
4,search_infm_params_text,large_string,0.0,0.6305
5,search_category,int64,0.0,0.0000


benchmark_items: 189212 строк


,column,type,nan_share,empty_str_share
0,item_title_raw,string,0.0000,0.0
1,item_rating_reviews_count,double,0.0614,0.0
2,item_rating,double,0.0932,0.0
3,item_price,"decimal128(27, 15)",0.0000,0.0
4,item_microcat_id,int64,0.0000,0.0
5,item_longitude,"decimal128(18, 15)",0.0000,0.0
6,item_location_id,int64,0.0000,0.0
7,item_latitude,"decimal128(17, 15)",0.0000,0.0
8,item_is_phone_hidden,bool,0.0000,0.0
9,item_is_message_forbidden,bool,0.0000,0.0


In [3]:
# Есть ли в train что-то похожее на время, сессию или id поиска
train_cols = pq.ParquetFile(DATA_DIR / "train.parquet").schema_arrow.names
print([c for c in train_cols if re.search(r"time|date|ts|session|search_id|event", c)])

[]


**Вывод 1.** Пропусков почти нет: в id, текстах объявлений и координатах их нет. Рейтинг и число отзывов пропущены у 4–9% объявлений.
Пустой `search_infm_params_text` у 33% строк train и у 63% запросов benchmark.
**Колонок времени, сессии или id поиска в train нет.** Значит, разбивать по времени нельзя, только по группам запросов.

## 2. Экземпляр запроса в train

In [4]:
# Всё, кроме описания: для разделов 2, 4, 5 оно не нужно.
TRAIN_COLS = [c for c in train_cols if c != "item_description_raw"]
train = load("train", TRAIN_COLS)
train["qtext"] = norm_text(train["search_query"])
train["qkey"] = query_key(train)
print("строк:", len(train))
print("уникальных текстов запроса:", train["qtext"].nunique())
print("уникальных экземпляров (ключ из 5 полей):", train["qkey"].nunique())
print("уникальных item_id:", train["item_id"].nunique())

строк: 497673
уникальных текстов запроса: 73873
уникальных экземпляров (ключ из 5 полей): 354305
уникальных item_id: 344825


In [5]:
g = train.groupby("qkey")
per_inst = pd.DataFrame({"rows": g.size(), "items": g["item_id"].nunique()})
bins = lambda s: s.clip(upper=6).value_counts(normalize=True).sort_index().rename(index={6: "6+"}).round(4)
display(pd.DataFrame({"строк на экземпляр": bins(per_inst["rows"]), "уник. item на экземпляр": bins(per_inst["items"])}))
print("среднее уник. item на экземпляр:", round(per_inst["items"].mean(), 3), "| макс:", per_inst["items"].max())

,строк на экземпляр,уник. item на экземпляр
1,0.8356,0.8497
2,0.0988,0.0950
3,0.0291,0.0264
4,0.0128,0.0111
5,0.0068,0.0057
6+,0.0168,0.0120


среднее уник. item на экземпляр: 1.318 | макс: 278


In [6]:
print("полных дублей строк (без описания):", train.duplicated(subset=TRAIN_COLS).sum())
print("дублей пары (экземпляр, item_id):", train.duplicated(subset=["qkey", "item_id"]).sum())
# Сколько экземпляров приходится на один текст: насколько текст запроса
# «размазан» по локациям и фильтрам. Это важно для группы разбиения.
inst_per_text = train.groupby("qtext")["qkey"].nunique()
print("экземпляров на текст:", inst_per_text.describe().round(2).to_dict())
print("доля текстов с 1 экземпляром:", round((inst_per_text == 1).mean(), 4))
top_texts = train.groupby("qtext").size().sort_values(ascending=False)
print("доля строк у топ-100 текстов:", round(top_texts.head(100).sum() / len(train), 4))
display(top_texts.head(15))

полных дублей строк (без описания): 30619
дублей пары (экземпляр, item_id): 30644
экземпляров на текст: {'count': 73873.0, 'mean': 4.8, 'std': 25.94, 'min': 1.0, '25%': 1.0, '50%': 1.0, '75%': 2.0, 'max': 1693.0}
доля текстов с 1 экземпляром: 0.6225
доля строк у топ-100 текстов: 0.251


qtext
маникюр                    6540
массаж                     5702
наращивание ресниц         5397
установка кондиционеров    4576
педикюр                    3436
сантехник                  3405
вывоз мусора               3288
эвакуатор                  3263
электрик                   3158
автоэлектрик               2790
покос травы                2484
вспашка земли              2316
манипулятор                2120
грузоперевозки газель      2034
натяжные потолки           2004
dtype: int64

In [7]:
# Популярность объявлений в train: сколько экземпляров выбирали одно и то же объявление
item_pop = train.groupby("item_id")["qkey"].nunique()
print(item_pop.describe(percentiles=[.5, .9, .99]).round(2).to_dict())

{'count': 344825.0, 'mean': 1.35, 'std': 0.86, 'min': 1.0, '50%': 1.0, '90%': 2.0, '99%': 5.0, 'max': 81.0}


**Вывод 2.** Экземпляров 354k, у 85% из них ровно одно выбранное объявление, в среднем 1.32. Хвост длинный, максимум 278.
Около 30.6k строк (6%) полностью дублируют друг друга: это повторные выборы той же пары «экземпляр, объявление». Для оценки берём множество уникальных item_id на экземпляр.
Тексты сильно неравномерны. У 62% текстов один экземпляр, а топ-100 текстов дают 25% строк («маникюр», «массаж»).
Объявление обычно выбирают в 1–2 экземплярах: популярность объявлений как сигнал слабая.

## 3. Есть ли объявления train в корпусе

In [8]:
items = load("benchmark_items")
print("корпус:", len(items), "| уникальных item_id:", items["item_id"].nunique())
train_items = train["item_id"].unique()
in_corpus = np.isin(train_items, items["item_id"].values)
print("уникальных item_id train в корпусе:", round(in_corpus.mean(), 4), f"({in_corpus.sum()} из {len(train_items)})")
print("строк train, чей item_id в корпусе:", round(train["item_id"].isin(items["item_id"]).mean(), 4))
print("доля корпуса, чей item_id встречается в train:", round(items["item_id"].isin(train["item_id"]).mean(), 4))

корпус: 189212 | уникальных item_id: 189212


уникальных item_id train в корпусе: 0.0526 (18142 из 344825)


строк train, чей item_id в корпусе: 0.0663


доля корпуса, чей item_id встречается в train: 0.0959


In [9]:
# Сверка по контенту: хэш считается по описанию, поэтому читаем его для train
# только для уникальных item_id и сразу хэшируем.
HASH_COLS = ["item_id", "item_title_raw", "item_description_raw", "item_infm_params_text", "item_location_id", "item_price"]
tr_items = load("train", HASH_COLS)
tr_items["h"] = content_hash(tr_items)
# Один item_id в train может встретиться с разным контентом, если объявление редактировали
h_per_id = tr_items.groupby("item_id")["h"].nunique()
print("item_id train с >1 вариантом контента:", (h_per_id > 1).sum())
tr_items = tr_items.drop_duplicates("item_id")[["item_id", "h"]]

items["h"] = content_hash(items)
print("уникальных хэшей в корпусе:", items["h"].nunique(), "из", len(items))
print("уникальных item train, чей хэш есть в корпусе:", round(tr_items["h"].isin(items["h"]).mean(), 4))

item_id train с >1 вариантом контента: 0


уникальных хэшей в корпусе: 189197 из 189212


уникальных item train, чей хэш есть в корпусе: 0.0526


In [10]:
# Для совпавших item_id: совпадает ли контент train и корпуса
m = tr_items.merge(items[["item_id", "h"]], on="item_id", suffixes=("_train", "_corpus"))
print("совпавших по id:", len(m), "| из них контент совпадает:", round((m["h_train"] == m["h_corpus"]).mean(), 4))

совпавших по id: 18142 | из них контент совпадает: 1.0


In [11]:
# Похожи ли объявления train (выбранные) на корпус. Важно для локального корпуса:
# если добавленные релевантные объявления отличаются от остального корпуса,
# ранкер может научиться находить именно их.
it_desc = load("benchmark_items", ["item_description_raw"])["item_description_raw"].fillna("").str.len()
tr_desc = load("train", ["item_id", "item_description_raw"]).drop_duplicates("item_id")
seen_mc = set(train["item_microcat_id"])
tri = train.drop_duplicates("item_id")
incorp = tri["item_id"].isin(items["item_id"])
def item_profile(d, desc_len):
    return pd.Series({
        "n": len(d),
        "доля без рейтинга": d["item_rating"].isna().mean(),
        "доля без числа отзывов": d["item_rating_reviews_count"].isna().mean(),
        "медиана цены": d["item_price"].median(),
        "длина заголовка": d["item_title_raw"].str.len().mean(),
        "длина infm_params": d["item_infm_params_text"].str.len().mean(),
        "медиана длины описания": desc_len.median(),
        "доля в microcat из train": d["item_microcat_id"].isin(seen_mc).mean(),
    })
tr_desc_len = tri[["item_id"]].merge(tr_desc, on="item_id")["item_description_raw"].fillna("").str.len()
display(pd.DataFrame({
    "train, нет в корпусе": item_profile(tri[~incorp.values], tr_desc_len[~incorp.values]),
    "train, есть в корпусе": item_profile(tri[incorp.values], tr_desc_len[incorp.values]),
    "корпус": item_profile(items, it_desc),
}).round(3))
del tr_desc

,"train, нет в корпусе","train, есть в корпусе",корпус
n,326683.000,18142.000,189212.000
доля без рейтинга,0.069,0.034,0.093
доля без числа отзывов,0.047,0.023,0.061
медиана цены,1000.000,1000.000,1000.000
длина заголовка,32.757,34.410,33.607
длина infm_params,1080.686,1132.172,973.897
медиана длины описания,846.000,1119.000,1054.000
доля в microcat из train,1.000,1.000,0.991


**Вывод 3.** **Объявления train почти не лежат в корпусе.** По item_id находится только 5.3% уникальных объявлений train, по хэшу контента ровно те же 5.3%.
При совпадении id контент совпадает всегда, и у одного id в train контент не меняется. Значит, id надёжны, а искусственный id не нужен.
Обратно: 9.6% корпуса встречаются в train. Признак «объявление есть в train» по правилам запрещён, я его не использую.
Ветка локального корпуса: benchmark_items плюс релевантные объявления P2/P3, которых в нём нет, со своими настоящими item_id.
Выбранные объявления систематически отличаются от корпуса. У них реже нет рейтинга, длиннее параметры и короче описание.
Это похоже на естественный отбор (выбирают объявления с отзывами), но локально это может завысить оценку. После разбиения проверю, насколько легко отличить добавленные объявления от корпуса.

## 4. Локация и категории

In [12]:
train["loc_match"] = train["item_location_id"] == train["search_location_id"]
display(train.groupby("search_is_delivery_search").agg(
    rows=("loc_match", "size"), loc_match_share=("loc_match", "mean")).round(4))

,rows,loc_match_share
search_is_delivery_search,,
0,497661,0.831
1,12,0.500


In [13]:
print("search_category:", train["search_category"].value_counts(normalize=True).round(4).head(10).to_dict())
print("item_category_id:", train["item_category_id"].value_counts(normalize=True).round(4).head(10).to_dict())
display(pd.crosstab(train["search_category"], train["item_category_id"], normalize="index").round(3))

search_category: {114: 0.9999, 0: 0.0001, 81: 0.0, 10: 0.0}
item_category_id: {114: 1.0, 27: 0.0, 40: 0.0, 81: 0.0, 19: 0.0, 28: 0.0, 112: 0.0, 10: 0.0}


item_category_id,10,19,27,28,40,81,112,114
search_category,,,,,,,,
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
10,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
81,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
114,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [14]:
# Насколько текст запроса определяет подкатегорию: для текстов с ≥5 строками
# доля самой частой microcat. Среднее взвешено по строкам (как в метрике по запросам).
tc = train.groupby(["qtext", "item_microcat_id"]).size()
tot = tc.groupby(level=0).sum()
top1 = tc.groupby(level=0).max() / tot
mask = tot >= 5
print("текстов с ≥5 строками:", mask.sum(), "| доля строк у них:", round(tot[mask].sum() / len(train), 4))
print("средняя доля топ-1 microcat (взвеш. по строкам):", round(np.average(top1[mask], weights=tot[mask]), 4))
print("квантили доли топ-1:", top1[mask].quantile([.1, .25, .5]).round(3).to_dict())
print("уникальных microcat в train:", train["item_microcat_id"].nunique(), "| в корпусе:", items["item_microcat_id"].nunique())

текстов с ≥5 строками: 11608 | доля строк у них: 0.8167
средняя доля топ-1 microcat (взвеш. по строкам): 0.852
квантили доли топ-1: {0.1: 0.5, 0.25: 0.667, 0.5: 0.875}
уникальных microcat в train: 212 | в корпусе: 752


In [15]:
print("item_category_id корпуса:", items["item_category_id"].value_counts(normalize=True).round(4).head(6).to_dict())
print("доля корпуса в microcat, встречавшихся в train:", round(items["item_microcat_id"].isin(seen_mc).mean(), 4))
print("строк train с search_category = 0:", (train["search_category"] == 0).sum())

item_category_id корпуса: {114: 0.9901, 19: 0.0019, 112: 0.0007, 40: 0.0006, 33: 0.0005, 10: 0.0005}
доля корпуса в microcat, встречавшихся в train: 0.9907
строк train с search_category = 0: 34


**Вывод 4.** Совпадение локации объявления и запроса 83%. Локация — сильный приор, но жёсткий фильтр потерял бы 17% ответов, поэтому только мягкий буст.
Доставки фактически нет: 12 строк в train и 0 в benchmark. Ветку с ослаблением локации для доставки не делаю.
Категория почти вырождена. В train и search_category, и item_category_id почти всегда 114 (услуги), в корпусе 99% объявлений в категории 114.
Подкатегорию текст запроса определяет хорошо: у частых текстов 85% выборов приходится на одну microcat. Классификатор запрос → microcat имеет смысл.
В train 212 microcat, в корпусе 752, но 99% корпуса лежит в microcat из train.

## 5. Benchmark против train

In [16]:
bq = load("benchmark_queries")
bq["qtext"] = norm_text(bq["search_query"])
bq["qkey"] = query_key(bq)
print("benchmark запросов:", len(bq), "| уникальных qkey:", bq["qkey"].nunique(), "| уникальных текстов:", bq["qtext"].nunique())
print("доля benchmark, чей текст есть в train:", round(bq["qtext"].isin(train["qtext"]).mean(), 4))
print("доля benchmark, чей экземпляр целиком есть в train:", round(bq["qkey"].isin(train["qkey"]).mean(), 4))
print("доля benchmark, чья локация есть в train:", round(bq["search_location_id"].isin(train["search_location_id"]).mean(), 4))

benchmark запросов: 2452 | уникальных qkey: 2452 | уникальных текстов: 2451


доля benchmark, чей текст есть в train: 0.3736


доля benchmark, чей экземпляр целиком есть в train: 0.044
доля benchmark, чья локация есть в train: 1.0


In [17]:
# Сравниваем benchmark с экземплярами train (не со строками): benchmark — по одному
# запросу на экземпляр, строки train перевзвешивают многоответные запросы.
tr_inst = train.drop_duplicates("qkey")
def profile(d):
    words = d["qtext"].str.split().str.len()
    return pd.Series({
        "n": len(d),
        "слов в запросе, среднее": words.mean(),
        "слов, медиана": words.median(),
        "доля пустых запросов": (d["qtext"] == "").mean(),
        "доля доставки": d["search_is_delivery_search"].mean(),
        "доля непустых infm_params": (norm_text(d["search_infm_params_text"]) != "").mean(),
    })
display(pd.DataFrame({"train (экземпляры)": profile(tr_inst), "benchmark": profile(bq)}).round(4))

,train (экземпляры),benchmark
n,354305.0000,2452.0000
"слов в запросе, среднее",2.4550,3.1974
"слов, медиана",2.0000,3.0000
доля пустых запросов,0.0000,0.0000
доля доставки,0.0000,0.0000
доля непустых infm_params,0.6479,0.3695


In [18]:
cmp = pd.DataFrame({
    "train": tr_inst["search_category"].value_counts(normalize=True),
    "benchmark": bq["search_category"].value_counts(normalize=True),
}).fillna(0).round(4).sort_values("train", ascending=False)
display(cmp.head(10))
loc = pd.DataFrame({
    "train": tr_inst["search_location_id"].value_counts(normalize=True),
    "benchmark": bq["search_location_id"].value_counts(normalize=True),
}).fillna(0).round(4).sort_values("train", ascending=False)
display(loc.head(10))
print("уникальных локаций: train", tr_inst["search_location_id"].nunique(), "| benchmark", bq["search_location_id"].nunique())

,train,benchmark
search_category,,
114,0.9999,0.9095
0,0.0001,0.0905
10,0.0000,0.0000
81,0.0000,0.0000


,train,benchmark
search_location_id,,
637640,0.0550,0.1342
653240,0.0384,0.0873
107620,0.0246,0.0485
633540,0.0195,0.0269
650400,0.0175,0.0306
621540,0.0150,0.0383
654070,0.0149,0.0171
641780,0.0145,0.0184
640860,0.0140,0.0204


уникальных локаций: train 2782 | benchmark 273


In [19]:
# Известность запроса в зависимости от его частоты в train: полезно для
# подбора доли «известных» в P3
freq = train.groupby("qtext")["qkey"].nunique()
bq["train_inst_cnt"] = bq["qtext"].map(freq).fillna(0).astype(int)
print(bq["train_inst_cnt"].clip(upper=20).value_counts(normalize=True).sort_index().round(4).to_dict())

{0: 0.6264, 1: 0.1036, 2: 0.0608, 3: 0.0396, 4: 0.0318, 5: 0.0204, 6: 0.0143, 7: 0.0143, 8: 0.0094, 9: 0.0073, 10: 0.0069, 11: 0.0077, 12: 0.0033, 13: 0.0045, 14: 0.002, 15: 0.0029, 16: 0.0029, 17: 0.0016, 18: 0.0012, 19: 0.0016, 20: 0.0375}


In [20]:
# Benchmark похож на выборку «один экземпляр на уникальный текст»:
# сравниваем с профилем уникальных текстов train, а не экземпляров.
tr_text = train.drop_duplicates("qtext")
display(pd.DataFrame({"train (уник. тексты)": profile(tr_text), "benchmark": profile(bq)}).round(4))
print("доля текстов train с одним экземпляром:", round((inst_per_text == 1).mean(), 4),
      "| доля benchmark с новым текстом:", round(1 - bq["qtext"].isin(train["qtext"]).mean(), 4))
b0 = bq[bq["search_category"] == 0]
print("benchmark с search_category=0:", len(b0), "| из них с непустым infm_params:", (norm_text(b0["search_infm_params_text"]) != "").sum())
display(b0[["search_query", "search_location_id"]].head(8))

,train (уник. тексты),benchmark
n,73873.0000,2452.0000
"слов в запросе, среднее",3.0990,3.1974
"слов, медиана",3.0000,3.0000
доля пустых запросов,0.0000,0.0000
доля доставки,0.0001,0.0000
доля непустых infm_params,0.5258,0.3695


доля текстов train с одним экземпляром: 0.6225 | доля benchmark с новым текстом: 0.6264
benchmark с search_category=0: 222 | из них с непустым infm_params: 0


,search_query,search_location_id
16,автомобиль бу под выкуп под такси,637640
24,установка реечного потолка в ванной,653240
28,ищу спарринг партнера по теннису,637640
40,экскурсия в чечню,647420
46,репетитор по веб разработке,653240
66,стеклянный шар с наклейкой,621540
75,регулировка петель двери,632660
82,доставка с европы,621540


**Вывод 5.** Текст 37.4% запросов benchmark встречается в train, а экземпляр целиком только у 4.4%. Все локации benchmark есть в train.
**Benchmark устроен как «один экземпляр на уникальный текст»:** на 2452 запроса приходится 2451 текст. Средняя длина совпадает с уникальными текстами train (3.20 и 3.10 слова), а не с экземплярами (2.46).
Доля новых текстов в benchmark 62.6%, а доля текстов train с единственным экземпляром 62.3%. Выборка «случайный текст, один его экземпляр» сама даёт нужную долю известных.
Сдвиги, которые разбиение не воспроизведёт:
- 9% benchmark имеют search_category = 0 (в train 34 строки). Среди них встречаются запросы не про услуги («автомобиль бу под выкуп»).
- infm_params непуст у 37% benchmark против 53% уникальных текстов train.
- Локации сильнее сосредоточены в крупных городах, например у 637640 доля 13% против 5.5% в train.
Эти срезы буду считать на P3 отдельно.

## 6. Дубли и уникальность id

In [21]:
print("item_id уникальны в корпусе:", items["item_id"].is_unique)
print("query_id уникальны в benchmark:", bq["query_id"].is_unique)
dup_sizes = items.groupby("h").size()
dups = dup_sizes[dup_sizes > 1]
print("групп одинакового контента:", len(dups), "| объявлений в них:", dups.sum(),
      f"({dups.sum() / len(items):.2%} корпуса) | макс. размер группы:", dups.max() if len(dups) else 0)
print("распределение размеров:", dups.clip(upper=10).value_counts().sort_index().to_dict())
print("для сравнения: уникальных заголовков", norm_text(items["item_title_raw"]).nunique())

item_id уникальны в корпусе: True
query_id уникальны в benchmark: True
групп одинакового контента: 12 | объявлений в них: 27 (0.01% корпуса) | макс. размер группы: 5
распределение размеров: {2: 11, 5: 1}


для сравнения: уникальных заголовков 118019


In [22]:
# Пример группы дублей глазами
if len(dups):
    h0 = dups.sort_values().index[-1]
    display(items.loc[items["h"] == h0, ["item_id", "item_title_raw", "item_location_id", "item_price", "item_microcat_id"]].head(5))

,item_id,item_title_raw,item_location_id,item_price,item_microcat_id
1024,4541d82f188196ad,Демонтаж Труб / Трубопроводов,637640,500.0,2301617
110848,e579df0ce02227b3,Демонтаж Труб / Трубопроводов,637640,500.0,2301617
122042,9ae0d281a23992a8,Демонтаж Труб / Трубопроводов,637640,500.0,2301617
130252,12ad0c1e46fd2da5,Демонтаж Труб / Трубопроводов,637640,500.0,2301617
143683,8388d77a8396163e,Демонтаж Труб / Трубопроводов,637640,500.0,2301617


**Вывод 6.** item_id уникальны в корпусе, query_id уникальны в benchmark.
Точных дублей контента почти нет: 12 групп, 27 объявлений. Признак «число дублей» не нужен.
Одинаковых заголовков при этом много: 118k уникальных на 189k объявлений. Поэтому хэш и строится по всему контенту.

## 7. Самопроверка A

In [23]:
ITEM_RE = re.compile(r"^[0-9a-f]{16}$")
assert len(train) == 497_673 and len(bq) == 2_452 and len(items) == 189_212
assert items["item_id"].map(lambda s: bool(ITEM_RE.match(s))).all(), "item_id корпуса не по regex"
bad_train_ids = (~train["item_id"].map(lambda s: bool(ITEM_RE.match(s)))).sum()
print("item_id train не по regex:", bad_train_ids)
assert bq["query_id"].str.len().eq(16).all()
assert items["item_id"].is_unique and bq["query_id"].is_unique
# «nan»/«None» как отдельное слово в текстах после fillna: признак, что где-то
# пропуск превратился в строку
NAN_RE = r"\b(?:nan|none)\b"
for df_, cols in [(train, ["search_query", "search_infm_params_text", "item_title_raw", "item_infm_params_text"]),
                  (bq, ["search_query", "search_infm_params_text"]),
                  (items, ["item_title_raw", "item_infm_params_text"])]:
    for c in cols:
        n = norm_text(df_[c]).str.contains(NAN_RE, regex=True).sum()
        print(f"{c}: строк со словом nan/none = {n}")
# Единичные «None» выше — исходные данные (например «Номер оригинальной запчасти None»),
# а не пропуск, превращённый в строку нашей предобработкой.
print("проверки A пройдены")

item_id train не по regex: 0


search_query: строк со словом nan/none = 0


search_infm_params_text: строк со словом nan/none = 0


item_title_raw: строк со словом nan/none = 0


item_infm_params_text: строк со словом nan/none = 3
search_query: строк со словом nan/none = 0
search_infm_params_text: строк со словом nan/none = 0


item_title_raw: строк со словом nan/none = 0


item_infm_params_text: строк со словом nan/none = 1
проверки A пройдены


## Выводы
1. Времени в train нет, поэтому разбиение идёт по группам текста запроса.
2. Объявления train почти не лежат в корпусе (5.3%). Локальный корпус = benchmark_items + релевантные объявления P2/P3 с настоящими id. Канал логов должен искать похожие объявления корпуса, а не те же самые.
3. Benchmark похож на выборку «один экземпляр на уникальный текст». Та же процедура на train естественно даёт долю известных текстов, как в benchmark (62.3% новых против 62.6%).
4. Локация: 83% совпадений, мягкий буст. Доставки нет. Категория вырождена (114), текст хорошо определяет microcat.
5. Сдвиги benchmark: 9% search_category = 0, реже infm_params, крупные города. Считать срезы на P3 отдельно.
6. Дублей объявлений нет, id чистые, проверки A пройдены.